# Quantum circuit simulator

A small state-vector simulator using NumPy arrays for states and gates. Basis states use MSB-first ordering, so $|011\rangle$ is vector index 3.

In [1]:
from collections import Counter

import numpy as np

from simulator import (
    CNOT, CZ, H, I, T, X, Y, Z,
    basis_state, measurement_probabilities, qft_matrix,
    sample_measurements, simulate,
)

np.set_printoptions(precision=3, suppress=True)

## One-qubit gates

In [2]:
zero = basis_state("0")
one = basis_state("1")
plus = (zero + one) / np.sqrt(2)

results = {
    "X|0>": simulate(zero, [X]),
    "Y|0>": simulate(zero, [Y]),
    "Z|1>": simulate(one, [Z]),
    "H|0>": simulate(zero, [H]),
    "T|+>": simulate(plus, [T]),
}
assert np.allclose(results["X|0>"], one)
assert np.allclose(results["H|0>"], plus)
results

{'X|0>': array([0.+0.j, 1.+0.j]),
 'Y|0>': array([0.+0.j, 0.+1.j]),
 'Z|1>': array([ 0.+0.j, -1.+0.j]),
 'H|0>': array([0.707+0.j, 0.707+0.j]),
 'T|+>': array([0.707+0.j , 0.5  +0.5j])}

## Two-qubit gates

Apply $H \otimes I$ and then CNOT to create a Bell state. CZ is also shown on $|++\rangle$.

In [3]:
bell = simulate(basis_state("00"), [np.kron(H, I), CNOT])
expected_bell = (basis_state("00") + basis_state("11")) / np.sqrt(2)
assert np.allclose(bell, expected_bell)

cz_on_plus_plus = simulate(np.kron(plus, plus), [CZ])
bell, cz_on_plus_plus

(array([0.707+0.j, 0.   +0.j, 0.   +0.j, 0.707+0.j]),
 array([ 0.5+0.j,  0.5+0.j,  0.5+0.j, -0.5+0.j]))

## Measurement

The Bell state has equal probability of producing `00` or `11`. A fixed seed keeps this example reproducible.

In [4]:
probabilities = measurement_probabilities(bell)
samples = sample_measurements(bell, shots=1_000, seed=7)
counts = {format(value, "02b"): count for value, count in Counter(samples).items()}

assert np.allclose(probabilities, [0.5, 0, 0, 0.5])
probabilities, counts

(array([0.5, 0. , 0. , 0.5]), {'11': 498, '00': 502})

## Three-qubit quantum Fourier transform

For $\omega=e^{2\pi i/8}$, the QFT maps $|j\rangle$ to $\frac{1}{\sqrt{8}}\sum_k \omega^{jk}|k\rangle$.

In [5]:
qft = qft_matrix(3)
omega = np.exp(2j * np.pi / 8)

output_000 = simulate(basis_state("000"), [qft])
output_011 = simulate(basis_state("011"), [qft])

assert np.allclose(output_000, np.ones(8) / np.sqrt(8))
assert np.allclose(output_011, omega ** (3 * np.arange(8)) / np.sqrt(8))
measurement_probabilities(output_011)

array([0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125])

In [6]:
superposition = (basis_state("000") + basis_state("011")) / np.sqrt(2)
output = simulate(superposition, [qft])
expected = (1 + omega ** (3 * np.arange(8))) / 4
probabilities = measurement_probabilities(output)

assert np.allclose(output, expected)
assert np.isclose(probabilities[0], 0.25)
assert np.isclose(probabilities[4], 0.0)
{format(index, "03b"): probability for index, probability in enumerate(probabilities)}

{'000': np.float64(0.25000000000000006),
 '001': np.float64(0.036611652351681595),
 '010': np.float64(0.12499999999999997),
 '011': np.float64(0.21338834764831854),
 '100': np.float64(7.252409594831223e-32),
 '101': np.float64(0.2133883476483183),
 '110': np.float64(0.12500000000000022),
 '111': np.float64(0.036611652351681394)}